# From Data to Decisions: A Deep Dive into Student Performance in Exams
### An End-to-End Exploratory Data Analysis, Feature Engineering & Business Analytics Project

## Project Overview

Education systems generate enormous amounts of data every single day as grades, demographics, socio-economic indicators, and behavioral signals yet very few institutions actually turn that data into **actionable insight**. 
This notebook takes the **Students Performance in Exams** dataset and treats it the way a data analyst embedded inside a school district, an ed-tech company, or a tutoring platform would: as a business problem in disguise.

We will move step by step from raw, unexamined data to a polished set of **data-driven recommendations** that a Head of Academics, a Curriculum Director, or a Student Success team could act on tomorrow morning.

## Dataset Description

The dataset (**`StudentsPerformance.csv`**) contains exam results for **1,000 students**, along with demographic and preparatory information:

| Column | Description |
|---|---|
| `gender` | Student's gender (male / female) |
| `race/ethnicity` | Ethnic group classification (Group A–E, anonymized) |
| `parental level of education` | Highest education level attained by a parent |
| `lunch` | Type of lunch program (`standard` or `free/reduced`) — a common socio-economic proxy |
| `test preparation course` | Whether the student completed a test-prep course (`completed` / `none`) |
| `math score` | Score out of 100 in Mathematics |
| `reading score` | Score out of 100 in Reading |
| `writing score` | Score out of 100 in Writing |

**Source:** [Kaggle — Students Performance in Exams](https://www.kaggle.com/datasets/spscientist/students-performance-in-exams)

## Business Context

Imagine this dataset originates from a school district (or an ed-tech platform partnering with several schools) that wants to understand **why some students consistently outperform others**, and more importantly **what interventions actually move the needle**. Resources for tutoring, test-prep subsidies, and counseling are limited, so the district needs to know **where to invest them for maximum impact**.

This reframes the analysis from "let's look at some scores" into a genuine business question: *how do we allocate a limited academic-support budget to close performance gaps as efficiently as possible?*

## Analysis Scope

This project follows a **Diagnostic & Predictive-Readiness Analytics** scope:
- **Diagnostic** — explaining *why* performance differs across student segments (gender, parental education, socio-economic proxy, test prep).
- **Predictive-readiness** — engineering features (total score, average score, pass/fail, performance bands) that would feed directly into a future predictive model, without building the model itself in this notebook.

## Project Objectives

1. Perform a rigorous, column-by-column exploratory data analysis.
2. Assess and document data quality issues, with clear (optional) cleaning code.
3. Engineer meaningful features that translate raw scores into business-relevant metrics.
4. Answer a set of concrete business questions using visual and statistical evidence.
5. Translate findings into an executive-ready set of recommendations.

## Key Business Questions

1. Which demographic and preparatory factors most influence student performance?
2. Does parental level of education correlate with student achievement?
3. Does completing a test-preparation course produce a measurable score lift?
4. Are there performance differences by gender, and if so, in which subjects?
5. Does the `lunch` socio-economic proxy relate to outcomes, and how strongly?
6. Which student segments are at the highest risk of failing or underperforming?
7. Which combination of factors best explains high achievement, and how could the district target interventions?

##  Expected Outcomes

By the end of this notebook, we will have:
- A fully documented, quality-checked dataset ready for modeling.
- A set of engineered features (Total Score, Average Score, Pass/Fail, Performance Band, Score Gap, Rank/Percentile).
- A rich set of interactive Plotly visualizations, each paired with a business interpretation.
- A concise executive summary with prioritized, actionable recommendations.


---

In [1]:
import numpy as np                     
import pandas as pd                     
import plotly.express as px             
import plotly.graph_objects as go       
from plotly.subplots import make_subplots   

# Display / styling configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# A consistent, professional color palette used throughout the notebook
COLOR_PALETTE = px.colors.qualitative.Set2
TEMPLATE = "plotly_white"

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Update this path if running outside of the Kaggle environment
DATA_PATH = "/kaggle/input/datasets/spscientist/students-performance-in-exams/StudentsPerformance.csv"

data = pd.read_csv(DATA_PATH)


In [3]:
data.head(15)

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75
5,female,group B,associate's degree,standard,none,71,83,78
6,female,group B,some college,standard,completed,88,95,92
7,male,group B,some college,free/reduced,none,40,43,39
8,male,group D,high school,free/reduced,completed,64,64,67
9,female,group B,high school,free/reduced,none,38,60,50


In [4]:
data.tail(15)

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
985,male,group A,high school,standard,none,57,51,54
986,female,group C,associate's degree,standard,none,40,59,51
987,male,group E,some high school,standard,completed,81,75,76
988,female,group A,some high school,free/reduced,none,44,45,45
989,female,group D,some college,free/reduced,completed,67,86,83
990,male,group E,high school,free/reduced,completed,86,81,75
991,female,group B,some high school,standard,completed,65,82,78
992,female,group D,associate's degree,free/reduced,none,55,76,76
993,female,group D,bachelor's degree,free/reduced,none,62,72,74
994,male,group A,high school,standard,none,63,63,62


In [5]:
# Dataset dimensions
print(f"Number of rows (students): {data.shape[0]}")
print(f"Number of columns (features): {data.shape[1]}")

Number of rows (students): 1000
Number of columns (features): 8


In [6]:
# Data types of each column
data.dtypes.to_frame(name="Data Type")

,Data Type
gender,object
race/ethnicity,object
parental level of education,object
lunch,object
test preparation course,object
math score,int64
reading score,int64
writing score,int64


> **Quick note:** All three score columns are numeric (`int64`), while the five remaining columns are categorical (`object`). This is exactly what we'd expect from an exam-results dataset, and it sets up the structure for the EDA that follows.


## Comprehensive Exploratory Data Analysis (EDA)

Before answering any business question, we need an intimate understanding of the dataset: its shape, its statistical fingerprint, and the story each individual column tells. This section is deliberately thorough every insight we surface here will directly inform the data quality assessment, feature engineering, and business analysis that follow.


### Dataset Overview
We start with a structural snapshot: shape, columns, dtypes, memory footprint, missing values, and duplicate records.


In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   gender                       1000 non-null   object
 1   race/ethnicity               1000 non-null   object
 2   parental level of education  1000 non-null   object
 3   lunch                        1000 non-null   object
 4   test preparation course      1000 non-null   object
 5   math score                   1000 non-null   int64 
 6   reading score                1000 non-null   int64 
 7   writing score                1000 non-null   int64 
dtypes: int64(3), object(5)
memory usage: 62.6+ KB


### Detect Nulls

In [8]:
# Missing values per column
missing_summary = pd.DataFrame({
    "Missing Count": data.isnull().sum(),
    "Missing %": (data.isnull().sum() / len(data) * 100).round(2)
})
missing_summary


,Missing Count,Missing %
gender,0,0.0
race/ethnicity,0,0.0
parental level of education,0,0.0
lunch,0,0.0
test preparation course,0,0.0
math score,0,0.0
reading score,0,0.0
writing score,0,0.0


### Detect Duplicates

In [9]:
print(f"Duplicate rows : {data.duplicated().sum()}")

Duplicate rows : 0


**Observation:** The dataset has **no missing values** and effectively no duplicate rows. This is a relatively "clean" dataset by real-world standards, which lets us focus most of our attention on distributional issues, naming consistency, and business-relevant feature creation rather than heavy imputation work.


### Descriptive Statistics

We separate numerical and categorical summaries, since they answer very different questions: numerical statistics describe the *shape and spread* of performance, while categorical statistics describe the *composition* of the student population.


In [10]:
# Numerical summary
numerical_cols = ["math score", "reading score", "writing score"]
data[numerical_cols].describe().round(2)


,math score,reading score,writing score
count,1000.00,1000.00,1000.00
mean,66.09,69.17,68.05
std,15.16,14.60,15.20
min,0.00,17.00,10.00
25%,57.00,59.00,57.75
50%,66.00,70.00,69.00
75%,77.00,79.00,79.00
max,100.00,100.00,100.00


**Observation:** Average scores across all three subjects hover in the high 60s, with reading and writing scoring slightly higher on average than math. All three subjects show a similar spread (standard deviation ~14–15 points), and the minimum values (as low as 0 in some subjects) hint at a small number of students who scored extremely poorly  worth flagging as potential outliers or at risk cases later.


In [11]:
# Categorical summary
categorical_cols = ["gender", "race/ethnicity", "parental level of education", "lunch", "test preparation course"]
data[categorical_cols].describe()


,gender,race/ethnicity,parental level of education,lunch,test preparation course
count,1000,1000,1000,1000,1000
unique,2,5,6,2,2
top,female,group C,some college,standard,none
freq,518,319,226,645,642


**Observation:** `race/ethnicity` has the most categories (5 groups), while the remaining categorical variables are binary or near binary. `top` and `freq` show us the most common category per column at a glance we'll quantify these fully in the column-by-column breakdown below.


### Column by Column Exploration

For every column, we examine its data type, cardinality, distribution, missing values, and critically its likely **business meaning**. This is where raw statistics start turning into a narrative.


In [12]:
def explore_column(col):
    print("=" * 70)
    print(f"COLUMN: {col}")
    print("=" * 70)
    print(f"Data type      : {data[col].dtype}")
    print(f"Unique values  : {data[col].nunique()}")
    print(f"Missing values : {data[col].isnull().sum()}")

    if data[col].dtype == "object":
        print("\nValue Counts:")
        print(data[col].value_counts())

        print("\nValue Counts (%):")
        print((data[col].value_counts(normalize=True) * 100).round(2))
    else:
        print("\nDistribution Summary:")
        print(data[col].describe().round(2))

In [13]:
explore_column("gender")

COLUMN: gender
Data type      : object
Unique values  : 2
Missing values : 0

Value Counts:
gender
female    518
male      482
Name: count, dtype: int64

Value Counts (%):
gender
female    51.8
male      48.2
Name: proportion, dtype: float64


In [58]:
fig = px.histogram(
    data,
    x="gender",
    color="gender",
    text_auto=True,
    title="Distribution of Gender"
)

fig.update_layout(showlegend=False)
fig.show()

In [15]:
explore_column("race/ethnicity")

COLUMN: race/ethnicity
Data type      : object
Unique values  : 5
Missing values : 0

Value Counts:
race/ethnicity
group C    319
group D    262
group B    190
group E    140
group A     89
Name: count, dtype: int64

Value Counts (%):
race/ethnicity
group C    31.9
group D    26.2
group B    19.0
group E    14.0
group A     8.9
Name: proportion, dtype: float64


In [59]:
fig = px.histogram(
    data,
    x="race/ethnicity",
    color="race/ethnicity",
    text_auto=True,
    title="Distribution of Race/Ethnicity"
)

fig.show()

In [60]:
explore_column("parental level of education")

COLUMN: parental level of education
Data type      : object
Unique values  : 6
Missing values : 0

Value Counts:
parental level of education
some college          226
associate's degree    222
high school           196
some high school      179
bachelor's degree     118
master's degree        59
Name: count, dtype: int64

Value Counts (%):
parental level of education
some college          22.6
associate's degree    22.2
high school           19.6
some high school      17.9
bachelor's degree     11.8
master's degree        5.9
Name: proportion, dtype: float64


In [61]:
fig = px.histogram(
    data,
    x="parental level of education",
    color="parental level of education",
    text_auto=True,
    title="Distribution of Parental Education"
)

fig.update_layout(xaxis_tickangle=-30)
fig.show()

In [19]:
explore_column("lunch")

COLUMN: lunch
Data type      : object
Unique values  : 2
Missing values : 0

Value Counts:
lunch
standard        645
free/reduced    355
Name: count, dtype: int64

Value Counts (%):
lunch
standard        64.5
free/reduced    35.5
Name: proportion, dtype: float64


In [62]:
fig = px.histogram(
    data,
    x="lunch",
    color="lunch",
    text_auto=True,
    title="Distribution of Lunch Type"
)

fig.update_layout(showlegend=False)
fig.show()

In [21]:
explore_column("test preparation course")

COLUMN: test preparation course
Data type      : object
Unique values  : 2
Missing values : 0

Value Counts:
test preparation course
none         642
completed    358
Name: count, dtype: int64

Value Counts (%):
test preparation course
none         64.2
completed    35.8
Name: proportion, dtype: float64


In [63]:
fig = px.histogram(
    data,
    x="test preparation course",
    color="test preparation course",
    text_auto=True,
    title="Distribution of Test Preparation Course"
)

fig.show()

In [23]:
explore_column("math score")

COLUMN: math score
Data type      : int64
Unique values  : 81
Missing values : 0

Distribution Summary:
count    1000.00
mean       66.09
std        15.16
min         0.00
25%        57.00
50%        66.00
75%        77.00
max       100.00
Name: math score, dtype: float64


In [64]:
fig = px.histogram(
    data,
    x="math score",
    nbins=20,
    marginal="box",
    title="Distribution of Math Scores"
)

fig.show()

In [25]:
explore_column("reading score")

COLUMN: reading score
Data type      : int64
Unique values  : 72
Missing values : 0

Distribution Summary:
count    1000.00
mean       69.17
std        14.60
min        17.00
25%        59.00
50%        70.00
75%        79.00
max       100.00
Name: reading score, dtype: float64


In [65]:
fig = px.histogram(
    data,
    x="reading score",
    nbins=20,
    marginal="box",
    title="Distribution of Reading Scores"
)

fig.show()

In [27]:
explore_column("writing score")

COLUMN: writing score
Data type      : int64
Unique values  : 77
Missing values : 0

Distribution Summary:
count    1000.00
mean       68.05
std        15.20
min        10.00
25%        57.75
50%        69.00
75%        79.00
max       100.00
Name: writing score, dtype: float64


In [66]:
fig = px.histogram(
    data,
    x="writing score",
    nbins=20,
    marginal="box",
    title="Distribution of Writing Scores"
)

fig.show()

#### Column Notes & Business Meaning

- **`gender`** — Near even split between male and female students. Business meaning: enables us to check for equity in outcomes across genders, a common fairness/DEI concern for education stakeholders.
- **`race/ethnicity`** — Five anonymized groups (A–E), with "Group C" typically the largest. Business meaning: proxy for demographic composition, useful for equity analysis, though the anonymization limits deeper socio-cultural interpretation.
- **`parental level of education`**  Six ordered categories from "some high school" to "master's degree." Business meaning: a well established socio-economic and academic support proxy; students with more highly educated parents often have more academic support at home.
- **`lunch`** - Binary: `standard` vs `free/reduced`. Business meaning: `free/reduced` lunch is a standard U.S. proxy for lower household income, making this one of the most business-relevant columns for equity-focused interventions.
- **`lunch`, `test preparation course`** — Both are actionable levers: a district cannot change a student's parental education, but it *can* expand access to free/reduced lunch support or test-prep courses.
- **`test preparation course`** — Binary: `completed` vs `none`. Business meaning: this is the one clearly *interventionable* variable in the dataset — the district controls whether it's offered and to whom.
- **`math score` / `reading score` / `writing score`** — Continuous outcome variables (0–100). Business meaning: these are the KPIs the district ultimately cares about improving.

**Interesting observation:** Because `test preparation course` and `lunch` are both variables a school district can directly act on, they will be central to the business recommendations at the end of this notebook unlike `gender`, `race/ethnicity`, or `parental level of education`, which are useful for *diagnosis* but not directly *actionable*.


## Data Quality Assessment
Even a well curated dataset like this one deserves a systematic quality audit. We check for the usual suspects: inconsistent text formatting, outliers and skew


In [29]:
print("LEADING / TRAILING WHITESPACE")
print("=" * 70)

whitespace_issues = {
    c: (data[c].astype(str) != data[c].astype(str).str.strip()).sum()
    for c in categorical_cols
}

for col, count in whitespace_issues.items():
    print(f"{col:<35}: {count}")

LEADING / TRAILING WHITESPACE
gender                             : 0
race/ethnicity                     : 0
parental level of education        : 0
lunch                              : 0
test preparation course            : 0


In [72]:
for col in categorical_cols:
    mask = data[col].astype(str) != data[col].astype(str).str.strip()

    if mask.sum() > 0:
        print(f"\n{col}")
        display(data.loc[mask, [col]])

In [69]:
print("CASE INCONSISTENCIES")
print("=" * 70)

case_issues = {
    c: data[c].astype(str).nunique() != data[c].astype(str).str.lower().nunique()
    for c in categorical_cols
}

for col, issue in case_issues.items():
    print(f"{col:<35}: {issue}")

CASE INCONSISTENCIES
gender                             : False
race/ethnicity                     : False
parental level of education        : False
lunch                              : False
test preparation course            : False


In [32]:
print("INVALID SCORE RANGES")
print("=" * 70)

invalid_scores = {
    c: int(((data[c] < 0) | (data[c] > 100)).sum())
    for c in numerical_cols
}

for col, count in invalid_scores.items():
    print(f"{col:<20}: {count}")

INVALID SCORE RANGES
math score          : 0
reading score       : 0
writing score       : 0


In [33]:
print("SKEWNESS")
print("=" * 70)

skewness = {
    c: round(data[c].skew(), 3)
    for c in numerical_cols
}

for col, value in skewness.items():
    print(f"{col:<20}: {value}")

SKEWNESS
math score          : -0.279
reading score       : -0.259
writing score       : -0.289


In [34]:
print("SKEWNESS INTERPRETATION")
print("=" * 70)

for col in numerical_cols:
    s = data[col].skew()

    if abs(s) < 0.5:
        interpretation = "Approximately Symmetric"
    elif abs(s) < 1:
        interpretation = "Moderately Skewed"
    else:
        interpretation = "Highly Skewed"

    print(f"{col:<20}: {s:.3f} ({interpretation})")

SKEWNESS INTERPRETATION
math score          : -0.279 (Approximately Symmetric)
reading score       : -0.259 (Approximately Symmetric)
writing score       : -0.289 (Approximately Symmetric)


In [35]:
# Outlier detection using the IQR method
def detect_outliers_iqr(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = series[(series < lower_bound) | (series > upper_bound)]
    return len(outliers), lower_bound, upper_bound

print("Outlier detection (IQR method, 1.5x rule):")
for col in numerical_cols:
    count, lower, upper = detect_outliers_iqr(data[col])
    print(f"  {col:15}: {count} outliers | valid range ≈ [{lower:.1f}, {upper:.1f}]")


Outlier detection (IQR method, 1.5x rule):
  math score     : 8 outliers | valid range ≈ [27.0, 107.0]
  reading score  : 6 outliers | valid range ≈ [29.0, 109.0]
  writing score  : 5 outliers | valid range ≈ [25.9, 110.9]


In [36]:
#flag (do not remove) potential low-score outliers for manual review
outlier_flags = pd.DataFrame({
col: (data[col] < detect_outliers_iqr(data[col])[1]) for col in numerical_cols})
data['flagged_for_review'] = outlier_flags.any(axis=1)
print(f"Rows flagged for review: {data['flagged_for_review'].sum()}")


Rows flagged for review: 12


In [37]:
# Create a DataFrame to store all unique outlier rows
all_outliers = pd.DataFrame()
for col in numerical_cols:
    count, lower, upper = detect_outliers_iqr(data[col])
    outlier_rows = data[
        (data[col] < lower) | (data[col] > upper)]

    all_outliers = pd.concat([all_outliers, outlier_rows])

# Remove duplicate rows (rows that are outliers in multiple columns)
all_outliers = all_outliers.drop_duplicates()
print(f"Total unique outlier rows: {len(all_outliers)}")

display(all_outliers)

Total unique outlier rows: 12


,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score,flagged_for_review
17,female,group B,some high school,free/reduced,none,18,32,28,True
59,female,group C,some high school,free/reduced,none,0,17,10,True
145,female,group C,some college,free/reduced,none,22,39,33,True
338,female,group B,some high school,free/reduced,none,24,38,27,True
466,female,group D,associate's degree,free/reduced,none,26,31,38,True
787,female,group B,some college,standard,none,19,38,32,True
842,female,group B,high school,free/reduced,completed,23,44,36,True
980,female,group B,high school,free/reduced,none,8,24,23,True
76,male,group E,some high school,standard,none,30,26,22,True
211,male,group C,some college,free/reduced,none,35,28,27,True


### Data Quality Findings Summary

| # | Issue | Found? | Notes |
|---|---|---|---|
| 1 | Missing values | ❌ None | Dataset is complete |
| 2 | Duplicate rows | ❌ None | No exact-duplicate students |
| 3 | Incorrect data types | ❌ None | Scores are correctly numeric |
| 4 | Leading/trailing whitespace | ❌ None found | Category text appears pre-cleaned |
| 5 | Case inconsistencies | ❌ None found | Categories consistently lower-case |
| 6 | Unexpected categories | ❌ None found | `gender`, `lunch`, `test preparation course` values match expectations |
| 7 | Invalid score ranges (outside 0–100) | ❌ None found | All scores within valid bounds |
| 8 | Outliers (IQR method) | ✅ **Yes, low-end outliers exist** | A handful of very low scores, especially in `math score` and `writing score` |
| 9 | Skewed distributions | ✅ **Yes, mild left (negative) skew** | Slightly more low-scoring outliers than high-scoring ones |

This is a genuinely clean dataset from a formatting standpoint there is no missing value imputation or text normalization work required. The one substantive issue is a small cluster of **very low outlier scores** (including some students scoring close to 0), which are far more likely to represent **real at-risk students** than data-entry errors. We treat them as a business signal rather than noise to be removed.


## Feature Engineering

Raw subject scores are useful, but a data analyst's real value lies in translating them into metrics that map directly onto business decisions. Below we engineer seven features, each with a clear rationale and business application.


### Total Score & Average Score
**Why useful:** Stakeholders rarely think in terms of three separate subject scores they want a single, comparable measure of overall academic performance per student.  
**Business value:** Enables ranking, benchmarking, and simple threshold-based reporting ("students averaging below 60").


In [38]:
data["total score"] = data["math score"] + data["reading score"] + data["writing score"]
data["average score"] = (data["total score"] / 3).round(2)

data[["math score", "reading score", "writing score", "total score", "average score"]].head()


,math score,reading score,writing score,total score,average score
0,72,72,74,218,72.67
1,69,90,88,247,82.33
2,90,95,93,278,92.67
3,47,57,44,148,49.33
4,76,78,75,229,76.33


### Pass/Fail Indicator
**Why useful:** A binary pass/fail flag (using a 50 point passing threshold per subject, a common academic convention) is the simplest possible KPI for a Student Success dashboard.  
**Business value:** Lets the district instantly quantify "how many students are failing at least one subject" — a number that translates directly into headcount for tutoring programs.


In [39]:
PASSING_THRESHOLD = 50
data["math pass"] = np.where(data["math score"] >= PASSING_THRESHOLD, "Pass", "Fail")
data["reading pass"] = np.where(data["reading score"] >= PASSING_THRESHOLD, "Pass", "Fail")
data["writing pass"] = np.where(data["writing score"] >= PASSING_THRESHOLD, "Pass", "Fail")

data["overall pass"] = np.where(
    (data["math pass"] == "Pass") & (data["reading pass"] == "Pass") & (data["writing pass"] == "Pass"),
    "Pass", "Fail"
)

data["overall pass"].value_counts()


overall pass
Pass    812
Fail    188
Name: count, dtype: int64

### Performance Category (Banding)
**Why useful:** Continuous average scores are hard to communicate in an executive summary.
Bucketing students into performance bands (`At Risk`, `Below Average`, `Average`, `Above Average`, `Excellent`) creates an intuitive segmentation.
**Business value:** Performance bands map directly onto tiered intervention programs — e.g., "At Risk" students get mandatory tutoring, "Excellent" students get advanced placement offers.


In [40]:
def performance_band(avg_score):
    if avg_score < 40:
        return "At Risk"
    elif avg_score < 60:
        return "Below Average"
    elif avg_score < 75:
        return "Average"
    elif avg_score < 90:
        return "Above Average"
    else:
        return "Excellent"

data["performance category"] = data["average score"].apply(performance_band)
band_order = ["At Risk", "Below Average", "Average", "Above Average", "Excellent"]
data["performance category"] = pd.Categorical(data["performance category"], categories=band_order, ordered=True)
data["performance category"].value_counts().reindex(band_order)


performance category
At Risk           30
Below Average    255
Average          391
Above Average    272
Excellent         52
Name: count, dtype: int64

### Score Gap (Subject Consistency Indicator)

**Why useful:** Some students perform very consistently across subjects, while others show a large gap (e.g., strong in math, weak in writing). The gap between max and min subject score captures this consistency.
**Business value:** Identifies students who might benefit from *subject specific* tutoring rather than general academic support a more cost efficient intervention.


In [41]:
data["score gap"] = data[["math score", "reading score", "writing score"]].max(axis=1) - data[["math score", "reading score", "writing score"]].min(axis=1)
#data["score gap"],value_counts()
data["score gap"].describe().round(2)


count    1000.00
mean        9.79
std         5.29
min         0.00
25%         6.00
50%         9.00
75%        13.00
max        28.00
Name: score gap, dtype: float64

### Rank

**Why useful:** Absolute scores don't tell a student (or administrator) how they compare to peers. Rank and percentile translate raw performance into relative standing.
**Business value:** Useful for scholarship eligibility, honor roll cutoffs, and percentile based reporting to parents/stakeholders.


In [42]:
data["rank"] = data["total score"].rank(ascending=False, method="min").astype(int)
data["percentile"] = (data["total score"].rank(pct=True) * 100).round(1)
data[["total score", "rank", "percentile"]].sort_values("rank").head()

,total score,rank,percentile
962,300,1,99.9
458,300,1,99.9
916,300,1,99.9
114,299,4,99.7
712,297,5,99.6


### Test Prep Effectiveness Flag
**Why useful:** Since `test preparation course` is one of the few *actionable* levers available to the district, it's worth engineering a feature that directly ties this variable to the average score, so its effect can be isolated cleanly in later visualizations.
**Business value:** Directly supports the ROI conversation around expanding (or not expanding) test-prep access.


In [43]:
prep_summary = data.groupby("test preparation course")["average score"].agg(["mean", "median", "std", "count"]).round(2)
prep_summary


,mean,median,std,count
test preparation course,,,,
completed,72.67,73.50,13.04,358
none,65.04,65.33,14.19,642


### 📌 Feature Engineering Summary

| Feature | Type | Business Purpose |
|---|---|---|
| `total score` | Numeric | Single combined performance measure |
| `average score` | Numeric | Normalized (0–100) performance measure |
| `math/reading/writing pass` | Categorical | Subject-level pass/fail tracking |
| `overall pass` | Categorical | District-wide pass-rate KPI |
| `performance category` | Ordinal | Tiered intervention targeting |
| `score gap` | Numeric | Subject-consistency / targeted-tutoring signal |
| `rank` / `percentile` | Numeric | Relative standing for reporting & recognition |

We now have a feature-rich dataset ready to answer the business questions posed at the start of this notebook.


## Business Understanding

### The Business Problem
A school district (or ed-tech partner) has a **limited academic support budget** and needs to decide how to allocate it: more test prep courses, subsidized lunch/nutrition programs, subject specific tutoring, or targeted counseling. Without data, these decisions default to intuition or politics rather than evidence.

### Why It Matters
Misallocated academic-support spending has real costs: students who need help don't get it, budget is wasted on interventions that don't move outcomes, and achievement gaps between demographic groups persist or widen — with downstream effects on graduation rates, college admissions, and long-term social mobility.

### Stakeholders
- **District Administrators / Superintendents** — own the budget and need defensible, evidence-based justification for spending decisions.
- **Curriculum & Instruction Teams** — need to know which subjects and student segments require curriculum-level intervention.
- **Student Success / Counseling Teams** — need a practical way to identify at-risk students early.
- **Parents & Students** — ultimate beneficiaries of better-targeted support.
- **Ed-Tech Partners (if applicable)** — need evidence that their product (e.g., test-prep courses) delivers measurable value.

### Business Hypotheses

1. **H1:** Students who complete a test-preparation course score significantly higher, on average, than those who do not.
2. **H2:** Students on the `free/reduced` lunch program (a socio-economic proxy) score lower, on average, than those on the `standard` plan.
3. **H3:** Higher parental education levels are associated with higher average student scores.
4. **H4:** Score patterns differ by gender, with the direction and magnitude of the difference varying by subject.
5. **H5:** A meaningful share of students fall into the "At Risk" or "Below Average" performance bands, representing a clear target population for intervention.

### From Business Objectives to Analytical Questions
| Business Objective | Analytical Question |
|---|---|
| Decide whether to expand test-prep access | Does test-prep completion produce a statistically and practically meaningful score lift? |
| Decide whether to expand lunch/nutrition support | Is there a measurable score gap between `standard` and `free/reduced` lunch students? |
| Identify students needing early intervention | What share of students fall into "At Risk" / "Below Average" bands, and what characterizes them? |
| Ensure equitable outcomes | Are there meaningful gender or ethnic-group score gaps that warrant investigation? |
| Prioritize subject-specific resourcing | Which subject shows the weakest overall performance and the widest spread? |


## Business Questions

With the business context established, we now formalize the specific questions this analysis will answer using the visualizations in the next section:
1. **Which factors most influence student performance?**
2. **Does parental education affect scores?**
3. **Does test preparation improve outcomes?**
4. **Are there gender-based performance differences, and in which subjects?**
5. **Which student groups are at the highest risk of poor performance?**
6. **Does the lunch program (socio-economic proxy) relate to performance?**
7. **How are the three subject scores related to one another — do strong math students also tend to be strong readers/writers?**
8. **What does the overall distribution of performance bands look like across the student population?**

Each question is answered with at least one purpose built visualization, followed by a markdown interpretation.


### Score Distributions
**Purpose:** Understand the overall shape of performance in each subject are scores roughly normal, skewed, bimodal?


In [44]:
fig = make_subplots(rows=1, cols=3, subplot_titles=("Math Score", "Reading Score", "Writing Score"))

for i, col in enumerate(numerical_cols, start=1):
    fig.add_trace(
        go.Histogram(x=data[col], name=col, marker_color=COLOR_PALETTE[i-1], nbinsx=30),
        row=1, col=i
    )

fig.update_layout(
    title_text="Distribution of Subject Scores",
    template=TEMPLATE,
    showlegend=False,
    height=420
)
fig.update_xaxes(title_text="Score")
fig.update_yaxes(title_text="Number of Students", row=1, col=1)
fig.show()


**Insight:** All three subjects show roughly bell-shaped distributions centered in the 65–70 range, with a mild left (negative) skew — a longer tail toward low scores than high scores. **Math** shows the widest spread and the most visible low-end tail, consistent with it typically being the subject where students struggle most. **Business implication:** Math is the strongest early candidate for targeted intervention resources.


### Score Spread by Subject
**Purpose:** Compare median, interquartile range, and outliers across the three subjects on a single, comparable axis.


In [45]:
score_long = data.melt(
    value_vars=numerical_cols,
    var_name="Subject",
    value_name="Score"
)

fig = px.box(
    score_long, x="Subject", y="Score", color="Subject",
    color_discrete_sequence=COLOR_PALETTE,
    title="Score Spread by Subject (Box Plot)",
    template=TEMPLATE,
    points="outliers"
)
fig.update_layout(showlegend=False)
fig.show()


**Insight:** Reading and writing scores have very similar medians and interquartile ranges, while math has a slightly lower median and a visibly longer lower tail with more low-end outliers. **Business implication:** This reinforces math as the subject with the largest concentration of low-performing students who could benefit from focused tutoring.


### Score Distribution by Test Preparation Status

**Purpose:** Go beyond the box plot's summary statistics to see the *full density shape* of scores for students who did vs. didn't complete test prep — directly testing **Hypothesis H1**.


In [46]:
summary = (
    data.groupby("test preparation course", as_index=False)["average score"]
    .agg(["mean", "std"])
    .reset_index()
)

fig = px.bar(
    summary,
    x="test preparation course",
    y="mean",
    color="test preparation course",
    error_y="std",
    text_auto=".1f",
    color_discrete_sequence=COLOR_PALETTE,
    title="Average Score by Test Preparation Course",
    labels={
        "test preparation course": "Test Preparation Course",
        "mean": "Average Score"
    },
    template=TEMPLATE
)

fig.update_layout(showlegend=False)
fig.show()

In [47]:
prep_lift = prep_summary.loc["completed", "mean"] - prep_summary.loc["none", "mean"]
print(f"Average score lift from completing test prep: +{prep_lift:.2f} points")


Average score lift from completing test prep: +7.63 points


**Insight:** Students who completed the test-preparation course show a visibly higher and tighter distribution of average scores than those who did not — a measurable, positive lift. **Business implication:** This directly supports **H1** and provides strong evidence for expanding access to (or making mandatory) the test-prep course, especially for students identified as at-risk.


### Average Scores by Gender and Subject

**Purpose:** Directly test **Hypothesis H4** — do score patterns differ by gender, and does the direction change by subject?


In [48]:
gender_subject = data.groupby("gender")[numerical_cols].mean().round(2).reset_index()
gender_subject_long = gender_subject.melt(id_vars="gender", var_name="Subject", value_name="Average Score")

fig = px.bar(
    gender_subject_long, x="Subject", y="Average Score", color="gender",
    barmode="group",
    color_discrete_sequence=COLOR_PALETTE,
    title="Average Score by Gender and Subject",
    template=TEMPLATE,
    text_auto=".1f"
)
fig.show()


**Insight:** Female students outperform male students on **reading and writing**, while male students outperform female students on **math**, on average. **Business implication:** Gender-based support should be subject-specific rather than blanket — e.g., targeted math confidence-building for female students and literacy-focused support for male students, rather than a one-size-fits-all program.


### Average Score by Parental Level of Education

**Purpose:** Test **Hypothesis H3** does parental education level correlate with student achievement?


In [49]:
edu_order = [
    "some high school", "high school", "some college",
    "associate's degree", "bachelor's degree", "master's degree"
]
edu_summary = data.groupby("parental level of education")["average score"].mean().reindex(edu_order).round(2).reset_index()

fig = px.bar(
    edu_summary, x="parental level of education", y="average score",
    color="average score", color_continuous_scale="Blues",
    title="Average Student Score by Parental Level of Education",
    labels={"parental level of education": "Parental Level of Education", "average score": "Average Score"},
    template=TEMPLATE,
    text_auto=".1f"
)
fig.update_xaxes(tickangle=-30)
fig.show()


**Insight:** There is a clear, largely monotonic upward trend: average student scores rise as parental education level increases, from "some high school" at the low end to "master's degree" at the high end. **Business implication:** This confirms **H3** and suggests that students of less-educated parents may lack academic support at home — a population that could benefit disproportionately from school-provided tutoring or homework support programs.


### Average Score by Lunch Type (Socio-Economic Proxy)

**Purpose:** Test **Hypothesis H2** does the `lunch` socio-economic proxy relate to performance?


In [50]:
lunch_summary = data.groupby("lunch")["average score"].agg(["mean", "median", "count"]).round(2).reset_index()

fig = px.bar(
    lunch_summary, x="lunch", y="mean", color="lunch",
    color_discrete_sequence=COLOR_PALETTE,
    title="Average Score by Lunch Program Type",
    labels={"lunch": "Lunch Program", "mean": "Average Score"},
    template=TEMPLATE,
    text_auto=".1f"
)
fig.update_layout(showlegend=False)
fig.show()


**Insight:** Students on the `standard` lunch plan score meaningfully higher, on average, than students on `free/reduced` lunch — a gap consistent with `lunch` acting as a socio-economic proxy. **Business implication:** This supports **H2** and strengthens the case for wraparound support (nutrition, after-school programs, family resource centers) targeted at lower-income students, not just academic tutoring in isolation.


### Math Score vs. Reading Score

**Purpose:** Explore the relationship between subjects do students who excel in math also tend to excel in reading?


In [51]:
fig = px.scatter(
    data, x="math score", y="reading score", color="test preparation course",
    color_discrete_sequence=COLOR_PALETTE,
    title="Math Score vs. Reading Score (colored by Test Prep Status)",
    labels={"math score": "Math Score", "reading score": "Reading Score"},
    template=TEMPLATE,
    opacity=0.6,
    trendline="ols"
)
fig.show()


**Insight:** There is a strong positive relationship between math and reading scores students who perform well in one subject tend to perform well in the other, suggesting a shared underlying "general academic ability" factor rather than fully independent subject-specific skills. Students who completed test prep (visually clustered toward the upper-right) tend to sit above the trend line more often. **Business implication:** Interventions that build general academic skills (study habits, time management) may have cross-subject benefits rather than needing to be siloed by subject.


### Correlation Heatmap

**Purpose:** Quantify the pairwise linear relationships between the three numeric scores.


In [52]:
corr_matrix = data[numerical_cols + ["average score"]].corr().round(2)

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale="RdBu",
    zmid=0,
    text=corr_matrix.values,
    texttemplate="%{text}",
    colorbar=dict(title="Correlation")
))
fig.update_layout(
    title="Correlation Matrix — Subject Scores",
    template=TEMPLATE,
    height=500
)
fig.show()


**Insight:** Reading and writing scores are the most strongly correlated pair (typically above 0.9), which makes intuitive sense — both are language-based skills. Math correlates strongly but slightly less tightly with both, confirming it as a more distinct skill domain. **Business implication:** Reading and writing curricula could reasonably be integrated or co-taught, while math likely needs a more dedicated, separate instructional strategy.


### Overall Pass/Fail Distribution

**Purpose:** Quantify, at a glance, what share of the student population is passing all three subjects — a headline KPI for any academic-success dashboard.


In [53]:
pass_counts = data["overall pass"].value_counts().reset_index()
pass_counts.columns = ["Status", "Count"]

fig = px.pie(
    pass_counts, names="Status", values="Count",
    color="Status",
    color_discrete_map={"Pass": COLOR_PALETTE[0], "Fail": COLOR_PALETTE[3] if len(COLOR_PALETTE) > 3 else COLOR_PALETTE[1]},
    title="Overall Pass Rate (All Three Subjects ≥ 50)",
    template=TEMPLATE,
    hole=0.4
)
fig.update_traces(textinfo="percent+label")
fig.show()


**Insight:** The large majority of students pass all three subjects at the 50 point threshold, but the failing minority still represents a meaningful headcount worth targeting with intervention resources — passing a low bar is not the same as being *well-prepared*. **Business implication:** A pure pass/fail KPI likely understates the true support need; performance bands (below) give a more nuanced picture.


### Performance Category Distribution

**Purpose:** Directly answer *"which student groups are at the highest risk of poor performance?"* using our engineered `performance category` feature.


In [54]:
band_counts = data["performance category"].value_counts().reindex(band_order).reset_index()
band_counts.columns = ["Performance Category", "Number of Students"]

fig = px.bar(
    band_counts, x="Performance Category", y="Number of Students",
    color="Performance Category",
    category_orders={"Performance Category": band_order},
    color_discrete_sequence=COLOR_PALETTE,
    title="Distribution of Students Across Performance Bands",
    template=TEMPLATE,
    text_auto=True
)
fig.update_layout(showlegend=False)
fig.show()


**Insight:** Most students fall into the "Average" and "Above Average" bands, but a non-trivial share land in "Below Average" or "At Risk" — this group represents the priority target population for tutoring and support programs. **Business implication:** This chart, filtered by demographic or socio-economic variables, becomes the exact operational tool a Student Success team would use to build a targeted-intervention roster.


### Score Relationships by Gender
**Purpose:** Combine the pairwise relationship view with a demographic lens to see whether the math–reading–writing relationship differs by gender.


In [76]:
fig = px.scatter(
    data,
    x="reading score",
    y="writing score",
    size="math score",
    color="gender",
    color_discrete_sequence=COLOR_PALETTE,
    title="Relationship Between Student Scores",
    template=TEMPLATE,
    opacity=0.7
)

fig.show()

**Insight:** The strong positive relationships between all subject pairs hold consistently for both genders — the *shape* of the relationship doesn't change by gender, even though (as shown earlier) the *average level* does. **Business implication:** Cross-subject predictive signals (e.g., using reading score to flag math risk) can be applied uniformly across genders without needing separate models.


### Who Are the "At Risk" Students?
**Purpose:** Move from *descriptive* to *diagnostic*: characterize the demographic composition of the highest-risk performance band so interventions can be targeted precisely.


In [56]:
at_risk = data[data["performance category"].isin(["At Risk", "Below Average"])]

print(f"Students in At Risk / Below Average bands: {len(at_risk)} ({len(at_risk) / len(data) * 100:.1f}% of student population)")
print()
print("Test preparation course completion within this group:")
print((at_risk["test preparation course"].value_counts(normalize=True) * 100).round(1))
print()
print("Lunch type within this group:")
print((at_risk["lunch"].value_counts(normalize=True) * 100).round(1))


Students in At Risk / Below Average bands: 285 (28.5% of student population)

Test preparation course completion within this group:
test preparation course
none         78.9
completed    21.1
Name: proportion, dtype: float64

Lunch type within this group:
lunch
free/reduced    54.4
standard        45.6
Name: proportion, dtype: float64


In [57]:
fig = px.sunburst(
    at_risk,
    path=["lunch", "test preparation course", "gender"],
    color="lunch",
    color_discrete_sequence=COLOR_PALETTE,
    title="Composition of At-Risk / Below-Average Students (Lunch → Test Prep → Gender)",
    template=TEMPLATE
)
fig.update_layout(height=550)
fig.show()


**Insight:** Within the at-risk population, students on `free/reduced` lunch and those who did **not** complete test prep are heavily over-represented relative to their share of the overall student body. **Business implication:** This is the single most actionable finding in the notebook — it points to a concrete, targetable intervention: prioritize free/reduced-lunch students who have not completed test prep as the **first cohort** for any new academic-support program, since they represent the highest concentration of risk.


---

## Insights Recap

Bringing together the findings from every section above:

| Business Question | Answer | Strength of Evidence |
|---|---|---|
| Which factors most influence performance? | Test prep completion, lunch type (socio-economic proxy), and parental education all show clear, consistent effects | Strong |
| Does parental education affect scores? | Yes — a largely monotonic upward trend from "some high school" to "master's degree" | Strong |
| Does test prep improve outcomes? | Yes — a clear, positive average score lift for students who completed it | Strong |
| Are there gender-based differences? | Yes, but subject-specific: female students lead in reading/writing, male students lead in math | Strong |
| Which groups are highest-risk? | Free/reduced-lunch students who have not completed test prep are disproportionately represented in the At Risk / Below Average bands | Strong |
| How do subjects relate to each other? | Reading and writing are most tightly correlated; math is a somewhat distinct skill domain | Strong |


## Final Recommendations

### Executive Summary

This analysis of 1,000 students' exam performance reveals that academic outcomes are shaped by a combination of controllable and uncontrollable factors. Among the controllable levers, **test-preparation course completion** shows the clearest, most actionable positive effect on scores. Socio-economic status (proxied by `lunch` type) and **parental education level** also correlate strongly with performance, though these are not directly within a school district's control. Gender differences exist but are subject-specific rather than uniform. Most critically, the intersection of **free/reduced lunch status and non-completion of test prep** identifies a clearly defined, disproportionately at-risk student segment — the natural first target for any new intervention budget.

### Major Insights

1. Test-prep completion produces a measurable, positive score lift — the strongest *actionable* lever in the dataset.
2. Socio-economic status (via the lunch proxy) and parental education both correlate strongly with performance, pointing to underlying equity gaps.
3. Gender gaps run in opposite directions by subject — math favors male students, reading/writing favor female students.
4. Math is the weakest-performing and most spread-out subject, and the most distinct from reading/writing in terms of skill correlation.
5. At-risk students are concentrated at the intersection of low socio-economic status and lack of test-prep access — not spread evenly across the population.

###  Business Recommendations

1. **Expand test-prep access, prioritizing free/reduced-lunch students.** This is the single most defensible investment given the clear score lift and its status as a directly controllable variable.
2. **Introduce math-specific tutoring resources**, given math's lower average score, wider spread, and higher concentration of low-end outliers relative to reading/writing.
3. **Design gender-aware, subject-specific support** rather than one-size-fits-all programs — literacy support skewed toward male students, math-confidence programs skewed toward female students.
4. **Build a "risk score" dashboard** using the `performance category`, `lunch`, and `test preparation course` fields to give counselors a live, prioritized outreach list.
5. **Investigate root causes of the parental-education gradient** (e.g., homework support access, take-home resources) to design interventions that don't rely on changing something the school cannot control.

### Suggested Next Steps

- Validate the test-prep effect with a **controlled or quasi-experimental study** (the current data is observational, so the effect could partly reflect self-selection — more motivated students may be more likely to both enroll in test prep and study harder generally).
- Build a **predictive model** (e.g., logistic regression or gradient boosting) using the engineered features here to proactively flag at-risk students before final exams.
- Bring in **longitudinal data** (multiple semesters/years) to see whether interventions produce sustained improvement rather than a one-time bump.
- Layer in **attendance and behavioral data**, if available, to enrich the risk profile beyond demographics and test prep status.

### Possible Future Analyses

- A/B test the causal impact of test-prep enrollment using a randomized subset of at-risk students.
- Cluster analysis (e.g., k-means on the three subject scores) to discover natural student archetypes beyond the linear performance bands used here.
- Cost-benefit modeling: estimate the budget required to bring all "At Risk" students to "Average" and compare it against projected long-term outcomes (graduation rates, college admission rates).

---

### Notebook Complete

This notebook took the Students Performance dataset from raw CSV to a full business narrative — data quality audit, engineered features, hypothesis-driven visual analysis, and a prioritized set of recommendations ready for stakeholder review. Feedback and forks are welcome!
